# Preparation of metrics for participant-day analyses

Primary analysis: documented non-wear removed; diary-defined sleep retained

Johannes Zauner

## Preface

This document collects the raw light exposure data for all three wearing positions through the `melidosData` package. This primary scenario removes documented non-wear outside diary-defined sleep while retaining sleep-time measurements. The data are preprocessed to

1.  An unaggregated set of light exposure data, both for the eye-level and chest-level wearing position where

- Values \> 100 000 lx are removed (set to `NA`)
- non-wear periods, that are not also sleep periods, are removed
- remove observations where not all three wearing positions are available

1.  A further processed set (from 1.) where

- hours with less than 50% data availability are removed
- days with less than 80% data availability (after the previous step) are removed

With the cleaned dataset, metrics per participant and day are calculated (with explicit exceptions). The resulting dataset is the basis for downstream inferential analyses.

## Setup

In [ ]:
#| label: setup
#| filename: Setup
library(tidyverse)
library(LightLogR)
library(melidosData)
library(gt)
library(here)
library(rlang)
library(cowplot)
source("../scripts/site_names.R")

## Importing data

### Light

In the first step, we import the unaggregated light eposure data for the glasses, chest, and wrist. These will be loaded from the [GitHub project page](https://github.com/MeLiDosProject). These data have been imported already, trimmed by trial dates, checked for irregular data and gaps. All of them run on a `10 second` interval.

In [ ]:
#| label: import-light-data
#| message: false
#| filename: Import light data
light_glasses <- load_data("light_glasses")
light_chest <- load_data("light_chest")
light_wrist <- load_data("light_wrist")

In [ ]:
#| label: count-participants
#| filename: Count participants and participant-days
light_glasses |> flatten_data() |> group_by(Id) |> n_groups()
light_chest |> flatten_data() |> group_by(Id) |> n_groups()
light_wrist |> flatten_data() |> group_by(Id) |> n_groups()
light_glasses |> flatten_data() |> group_by(Id) |> add_Date_col(group.by = TRUE) |> n_groups()
light_chest |> flatten_data() |> group_by(Id) |> add_Date_col(group.by = TRUE) |> n_groups()
light_wrist |> flatten_data() |> group_by(Id) |> add_Date_col(group.by = TRUE) |> n_groups()

### Sleep

The sleep data comes from a morning sleep diary. They contain both sleep and wake times.

In [ ]:
#| label: import-sleep-data
#| message: false
#| filename: Import sleep data
sleepdiary <- load_data("sleepdiaries")

### Wear log

The non-wear data comes from an app-based wear log that participants filled in whenever they removed or put on the device(s).

In [ ]:
#| label: import-wearlog-data
#| message: false
#| filename: Import wearlog data
wearlog <- load_data("wearlog")

### Combine wearing positions

- Only a selection of variables will be kept, namely `MEDI` (melanopic EDI) and `LIGHT` (photopic illuminance).

- Other contextual variables that are kept are `Id`, `Datetime`, and `position`.

- We further remove the `MPI` site from the glasses dataset, as no chest or wrist[1] data were collected.

- Then we will combine the observations of the wearing positions.

- Lastly, we will calculate the photoperiod information for each site.

[1] Wrist data were collected but with a different device (`ActTrust` instead of `ActLumus`). Thus, it will not be considered here.

In [ ]:
#| label: remove-mpi-site
#| filename: Remove MPI site
light_glasses$MPI <- NULL

In [ ]:
#| label: align-combine-positions
#| filename: Select relevant columns, combine positions, add photoperiod
light <-
  imap(
    light_glasses,
    \(data, idx) data |>
      select(Id, Datetime, MEDI, LIGHT) |> 
      data2reference(light_chest[[idx]], Reference.column = MEDI_chest) |> 
      data2reference(light_chest[[idx]],
                     Data.column = LIGHT,
                     Reference.column = LIGHT_chest) |> 
      data2reference(light_wrist[[idx]], Reference.column = MEDI_wrist) |>
      data2reference(light_wrist[[idx]],
                     Data.column = LIGHT,
                     Reference.column = LIGHT_wrist) |>
      add_photoperiod(melidos_coordinates[[idx]]) |> 
      rename(MEDI_glasses = MEDI, LIGHT_glasses = LIGHT)
  )

> Note: the displayed warnings refer to the fact that participant `FUSPCEU_S014` had a measurement interval of 60 seconds on the glasses, compared to the 10 seconds of everyone else (and both the other wearing positions). Thus, six measurement values from the chest and wrist positions map to one from the glasses. This is not deemed problematic, as it is only one participant in one site. The last of the six respective chest and wrist measurements will be used for comparison.

### Remove instances with less than all wearing positions

In [ ]:
#| label: enforce-concurrency
#| filename: Remove instances with less than all wearing positions
light <- 
  light |> 
  map(
    \(data) data |> 
              mutate(
                across(MEDI_glasses:LIGHT_wrist,
                \(x) {
                ifelse(is.na(MEDI_glasses) | is.na(MEDI_chest) | is.na(MEDI_wrist),
                       NA, x)
                }
              )
              )
  )

### Combine light exposure data with log and diary data

For the sleepdiary, a selection of sleep and wake times will be added. As the variable `sleepprep` (preparation for sleep) is the more reasonable time indicator for when the device positions are set for the night, this variable will be used instead of the calculated time of `sleep` (`sleepprep` + `sleepdelay`). For the wearlog, the start and end times of a removal, as well as the type (`state`) of removal will be used.

#### Preparation

In [ ]:
#| label: prepare-sleep-wearlog
#| filename: Prepare sleepdiaries and wearlog for merging
#| message: false
sleepdiary_adj <-
  sleepdiary |>
  map(\(x) x |>
        select(Id, sleepprep, wake) |>
        group_by(Id) |>
        pivot_longer(-Id, names_to = "sleep", values_to = "Datetime") |>
        sc2interval(Statechange.colname = sleep, starting.state = "wake") |>
        sleep_int2Brown(sleep.state = "sleepprep", Brown.day = "wake", 
                        Brown.evening = "pre-sleep", Brown.night = "sleep") |>
        mutate(sleep = case_when(is.na(sleep) & State.Brown == "pre-sleep" ~ "wake",
                                 .default = sleep))
  )

wearlog_adj <-
  wearlog |>  
    map(\(x) x |> select(Id, start, end, wear = state))

#### Combination

In [ ]:
#| label: merge-sleep-wearlog
#| filename: Add sleepdiaries and wearlog to light
light <- 
light |> 
  imap(\(x, idx) x |>
    add_states(sleepdiary_adj[[idx]], start = Interval, end = Interval) |>
    add_states(wearlog_adj[[idx]])
  )

``` r
#| label: tbl-data-overview-before-step1
#| filename: Overview of data non-wear and out-of-range removal
light |> 
  structure(class = "melidos_data") |> 
  flatten_data() |> 
  group_by(site, Id) |> 
  add_Date_col(group.by = TRUE) |>
  filter_out(any(all(is.na(MEDI_glasses)), all(is.na(MEDI_chest)), all(is.na(MEDI_wrist)))) |> 
  ungroup(Date) |> 
  distinct(Date) |>
  group_by(site) |> 
  count() |> 
  ungroup() |> 
  site_conv_mutate() |> 
  gt() |> 
  cols_label(site = "Site",
             n = "Participant-days") |> 
  grand_summary_rows(
    n, fns = sum ~ sum(.)
  ) |> 
  tab_header("Summary of participant_days")
```

## Preprocessing step 1

In this step, we will remove non-wear instances that are neither marked as sleep, nor fall into a sleep window (sleepdiary). We will also remove instances ≥ 1.0\*10^5 lx melanopic EDI.

In [ ]:
#| label: preprocessing-step1
#| filename: remove non-wear and out-of-range measurements
light <- 
light |> 
  imap(\(x, idx) x |> 
    mutate(across(c(starts_with("MEDI"), starts_with("LIGHT")),
                  \(y) replace_when(y, 
                               wear == "off" & (State.Brown != "sleep" | is.na(State.Brown)) ~ NA,
                               y >= 100000 ~ NA
                               )),
           across(MEDI_glasses:LIGHT_wrist,
                \(z) {
                ifelse(is.na(MEDI_glasses) | is.na(MEDI_chest) | is.na(MEDI_wrist),
                       NA, z)
                }
              )
  )
  )

In [ ]:
#| label: flatten-to-dataframe
#| filename: Flatten structure from list to data.frame
light_flat <- 
  structure(light, class = "melidos_data") |> flatten_data() |> group_by(site, Id)

``` r
#| label: tbl-data-overview-after-step1
#| filename: Overview of data non-wear and out-of-range removal
light_flat |> 
  drop_na(MEDI_glasses) |> 
  add_Date_col() |> 
  distinct(Date) |>
  group_by(site) |> 
  count() |> 
  ungroup() |> 
  site_conv_mutate() |> 
  gt() |> 
  cols_label(site = "Site",
             n = "Participant-days") |> 
  grand_summary_rows(
    n, fns = sum ~ sum(.)
  ) |> 
  tab_header("Summary of remaining participant days after non-wear and out-of-range removal")
```

## Preprocessing step 2

Here we further process the data: - hours with less than 50% data availability are removed - days with less than 80% data availability (after the previous step) are removed

In [ ]:
#| label: preprocessing-step2
#| filename: remove hours and days with too few data
light_fin <- 
  light_flat |> 
  cut_Datetime(unit = "1 hour", group_by = TRUE, type = "floor") |> 
  remove_partial_data(MEDI_glasses, threshold.missing = 0.5) |> 
  ungroup(Datetime.rounded) |> 
  select(-Datetime.rounded) |> 
  add_Date_col(group.by = TRUE) |> 
  gap_handler(full.days = TRUE) |> 
  remove_partial_data(MEDI_glasses, threshold.missing = 0.2) |> 
  ungroup(Date)

``` r
#| label: tbl-data-overview-after-step2
#| filename: Overview of data after preprocessing
light_fin |> 
  distinct(Date) |>
  group_by(site) |> 
  count() |> 
  ungroup() |> 
  site_conv_mutate() |> 
  gt() |> 
  cols_label(site = "Site",
             n = "Participant-days") |> 
  grand_summary_rows(
    n, fns = sum ~ sum(.)
  ) |> 
  tab_header("Summary of remaining participant days after preprocessing")
```

## Calculate metrics

In [ ]:
#| label: datetime-conversion-helper
#| filename: Create convenience function for converting datetime to time to double
datetime_2_numeric <- function(x) {
  x |> 
    mutate(
      across(
        where(is.POSIXct),
        \(x) x |> hms::as_hms() %>% as.numeric()
      )
    )
}

In [ ]:
#| label: prepare-metrics-data
#| filename: Prepare metrics data
metrics_data <-
  light_fin |> 
  distinct(site, Id, Datetime, .keep_all = TRUE) |> 
    pivot_longer(MEDI_glasses:LIGHT_wrist,
                 names_sep = "_",
                 names_to = c("metric", "position")) |> 
    pivot_wider(values_from = value, names_from = metric)

In [ ]:
#| label: calculate-daily-metrics
#| filename: Calculate by-day metrics
#| warning: false
metrics <- 
  metrics_data |> 
  group_by(site, Id, Date, position) |> 
        summarize(
          duration_above_threshold(
            MEDI, Datetime, "above", 10, na.rm = TRUE, as.df = TRUE),
          duration_above_threshold(
            MEDI, Datetime, "above", 250, na.rm = TRUE, as.df = TRUE),
          duration_above_threshold(
            MEDI, Datetime, "above", 1000, na.rm = TRUE, as.df = TRUE),
          period_above_threshold(
            MEDI, Datetime, "above", 10, na.rm = TRUE, as.df = TRUE),
          period_above_threshold(
            MEDI, Datetime, "above", 250, na.rm = TRUE, as.df = TRUE),
          period_above_threshold(
            MEDI, Datetime, "above", 1000, na.rm = TRUE, as.df = TRUE),
          pulses_above_threshold(
            MEDI, Datetime, threshold = 250, na.rm = TRUE, as.df = TRUE
          )|>
            datetime_2_numeric(),
          pulses_above_threshold(
            MEDI, Datetime, threshold = 1000, na.rm = TRUE, as.df = TRUE
          )|>
            datetime_2_numeric(),
          bright_dark_period(
                  MEDI |> log_zero_inflated(),
                  Datetime, "brightest", "10 hours",
                  as.df = TRUE, na.rm = TRUE
                  ) %>%
            datetime_2_numeric(),
          bright_dark_period(
                  MEDI |> log_zero_inflated(),
                  Datetime, "darkest", "10 hours", as.df = TRUE,
                  loop = TRUE, na.rm = TRUE
                  ) %>%
            datetime_2_numeric(),
          timing_above_threshold(
              MEDI, Datetime, "above", 10, as.df = TRUE) |>
            datetime_2_numeric(),
          timing_above_threshold(MEDI, Datetime, "above", 250, as.df = TRUE) |>
            datetime_2_numeric(),
          frequency_crossing_threshold(MEDI, 250, na.rm = TRUE, as.df = TRUE),
          timing_above_threshold(
              MEDI, Datetime, "above", 1000, as.df = TRUE) |>
            datetime_2_numeric(),
          barroso_lighting_metrics(
            MEDI, Datetime, loop = TRUE, na.rm = TRUE, as.df = TRUE
             ),
          centroidLE(MEDI, Datetime, na.rm = TRUE, as.df = TRUE) |>
            datetime_2_numeric(),
          disparity_index(MEDI, TRUE, TRUE),
          midpointCE(MEDI, Datetime, TRUE, TRUE)|>
            datetime_2_numeric(),
          mean_MEDI = mean(MEDI |> log_zero_inflated(), na.rm = TRUE),
          nvRD = nvRD(MEDI, LIGHT, Datetime) |> mean(na.rm = TRUE),
          dose(MEDI, Datetime, na.rm = TRUE, as.df = TRUE),
          MDER = median(MEDI / LIGHT, na.rm = TRUE),
          .groups = "drop",
        ) %>%
        mutate(across(where(is.duration), as.numeric)) %>% 
        pivot_longer(cols = -c(site, Id, Date, position), names_to = "metric")

In [ ]:
#| label: calculate-multiday-metrics
#| filename: Calculate by-day metrics
#| warning: false
metrics2 <- 
  metrics_data |> 
  group_by(site, Id, position) |> 
  summarize(
    interdaily_stability(MEDI |> log_zero_inflated(), 
                        Datetime, na.rm = TRUE, as.df = TRUE),
    intradaily_variability(MEDI |> log_zero_inflated(), 
                           Datetime, na.rm = TRUE, as.df = TRUE),
          ) |> 
  pivot_longer(cols = -c(site, Id, position), names_to = "metric")

In [ ]:
#| label: combine-all-metrics
#| filename: combine metrics
metrics <- 
        bind_rows(
          metrics,
          metrics2
          )
rm(metrics2)

## 1-hour-minute values

For the nonlinear explorative analysis, we require 1-hour-values.

In [ ]:
#| label: aggregate-hourly
time_data <- 
metrics_data |> 
  group_by(site, Id, Date, position) |> 
  aggregate_Datetime(
    "1 hour",
    type = "floor",
    numeric.handler = \(x) x |> mean(na.rm = TRUE),
    geo.MEDI = MEDI |> log_zero_inflated() |> mean(na.rm = TRUE) |> exp_zero_inflated()
  )|>
  add_Date_col(group.by = TRUE) |> 
  mutate(static = all(MEDI == MEDI[1])) |> 
  filter_out(static) |> 
  select(-static) |> 
  add_Time_col() |> 
  ungroup() |> 
  mutate(Time = as.numeric(Time)/3600 + 0.5,
         across(c(site, Id, sleep, State.Brown, wear, photoperiod.state),
                fct),
         Date = factor(Date),
         Id_date = interaction(Id, Date),
         lzMEDI = log_zero_inflated(MEDI),
         photoperiod = (dusk - dawn) |> as.numeric()
  ) |> 
  group_by(Id, position) |> 
  mutate(AR.start = ifelse(row_number() == 1, TRUE, FALSE)) |> 
  ungroup()

## Export metrics

In [ ]:
#| label: export-data
#| filename: Export metric data
save(metrics, file = here("data/prepared_metrics.RData"))
save(time_data, file = here("data/prepared_time_data.RData"))

## Session info

In [ ]:
#| label: session-info
sessionInfo()